In [3]:
from pathlib import Path
import os
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from PIL import Image
from torchvision import transforms
import timm

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

In [4]:
ROOT = Path("./")
META_PATH = ROOT / "metadata.csv"

classes = ["akiec", "bcc", "bkl", "df", "mel", "nv", "vasc"]
label2idx = {c: i for i, c in enumerate(classes)}
idx2label = {i: c for c, i in label2idx.items()}

OUTPUT_DIR = Path("outputs_inceptionresnetv2_metadata_weighted")
OUTPUT_DIR.mkdir(exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [5]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

set_seed(42)

In [6]:
df = pd.read_csv(META_PATH)

print(df.head())
print(df.columns)
print(df.shape)
print(df["dx"].value_counts())
print(df[["age", "sex", "localization"]].isna().sum())

     lesion_id      image_id   dx dx_type   age   sex localization  \
0  HAM_0000118  ISIC_0027419  bkl   histo  80.0  male        scalp   
1  HAM_0000118  ISIC_0025030  bkl   histo  80.0  male        scalp   
2  HAM_0002730  ISIC_0026769  bkl   histo  80.0  male        scalp   
3  HAM_0002730  ISIC_0025661  bkl   histo  80.0  male        scalp   
4  HAM_0001466  ISIC_0031633  bkl   histo  75.0  male          ear   

        dataset  
0  vidir_modern  
1  vidir_modern  
2  vidir_modern  
3  vidir_modern  
4  vidir_modern  
Index(['lesion_id', 'image_id', 'dx', 'dx_type', 'age', 'sex', 'localization',
       'dataset'],
      dtype='object')
(10015, 8)
dx
nv       6705
mel      1113
bkl      1099
bcc       514
akiec     327
vasc      142
df        115
Name: count, dtype: int64
age             57
sex              0
localization     0
dtype: int64


In [7]:
def get_image_path(row):
    cls = row["dx"]
    image_id = row["image_id"]
    path = ROOT / cls / "enhanced" / f"{image_id}.jpg"
    if path.exists():
        return str(path)
    return None

df["image_path"] = df.apply(get_image_path, axis=1)

print("missing images:", df["image_path"].isna().sum())

missing_df = df[df["image_path"].isna()]
print(missing_df[["image_id", "dx", "image_path"]].head())

df = df.dropna(subset=["image_path"]).reset_index(drop=True)

print(df.shape)
print(df["dx"].value_counts())

missing images: 0
Empty DataFrame
Columns: [image_id, dx, image_path]
Index: []
(10015, 9)
dx
nv       6705
mel      1113
bkl      1099
bcc       514
akiec     327
vasc      142
df        115
Name: count, dtype: int64


In [8]:
df["label"] = df["dx"].map(label2idx)

print(df[["image_id", "dx", "label", "age", "sex", "localization", "image_path"]].head())
print(df["label"].value_counts().sort_index())

       image_id   dx  label   age   sex localization  \
0  ISIC_0027419  bkl      2  80.0  male        scalp   
1  ISIC_0025030  bkl      2  80.0  male        scalp   
2  ISIC_0026769  bkl      2  80.0  male        scalp   
3  ISIC_0025661  bkl      2  80.0  male        scalp   
4  ISIC_0031633  bkl      2  75.0  male          ear   

                      image_path  
0  bkl/enhanced/ISIC_0027419.jpg  
1  bkl/enhanced/ISIC_0025030.jpg  
2  bkl/enhanced/ISIC_0026769.jpg  
3  bkl/enhanced/ISIC_0025661.jpg  
4  bkl/enhanced/ISIC_0031633.jpg  
label
0     327
1     514
2    1099
3     115
4    1113
5    6705
6     142
Name: count, dtype: int64


In [9]:
train_df, val_df = train_test_split(
    df,
    test_size=0.1,
    stratify=df["dx"],
    random_state=42
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print("Train size:", len(train_df))
print("Val size:", len(val_df))

print("\nTrain distribution:")
print(train_df["dx"].value_counts())

print("\nVal distribution:")
print(val_df["dx"].value_counts())

Train size: 9013
Val size: 1002

Train distribution:
dx
nv       6034
mel      1002
bkl       989
bcc       463
akiec     294
vasc      128
df        103
Name: count, dtype: int64

Val distribution:
dx
nv       671
mel      111
bkl      110
bcc       51
akiec     33
vasc      14
df        12
Name: count, dtype: int64


In [10]:
train_df = train_df.copy()
val_df = val_df.copy()

# age 缺失用训练集 median 填充
age_median = train_df["age"].median()
train_df["age"] = train_df["age"].fillna(age_median)
val_df["age"] = val_df["age"].fillna(age_median)

# sex/localization 缺失填 unknown
for col in ["sex", "localization"]:
    train_df[col] = train_df[col].fillna("unknown").astype(str)
    val_df[col] = val_df[col].fillna("unknown").astype(str)

# age 标准化
scaler = StandardScaler()
train_age = scaler.fit_transform(train_df[["age"]])
val_age = scaler.transform(val_df[["age"]])

# sex/localization one-hot
try:
    encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    encoder = OneHotEncoder(handle_unknown="ignore", sparse=False)

train_cat = encoder.fit_transform(train_df[["sex", "localization"]])
val_cat = encoder.transform(val_df[["sex", "localization"]])

# 拼接 metadata
train_meta = np.concatenate([train_age, train_cat], axis=1).astype("float32")
val_meta = np.concatenate([val_age, val_cat], axis=1).astype("float32")

metadata_dim = train_meta.shape[1]

print("metadata_dim:", metadata_dim)
print("train_meta shape:", train_meta.shape)
print("val_meta shape:", val_meta.shape)
print("metadata categories:", encoder.categories_)

metadata_dim: 19
train_meta shape: (9013, 19)
val_meta shape: (1002, 19)
metadata categories: [array(['female', 'male', 'unknown'], dtype=object), array(['abdomen', 'acral', 'back', 'chest', 'ear', 'face', 'foot',
       'genital', 'hand', 'lower extremity', 'neck', 'scalp', 'trunk',
       'unknown', 'upper extremity'], dtype=object)]


In [11]:
num_classes = len(classes)

class_counts = train_df["label"].value_counts().sort_index().values
N = class_counts.sum()
C = num_classes

class_weights = N / (C * class_counts)
class_weights = torch.tensor(class_weights, dtype=torch.float32)

for cls, count, weight in zip(classes, class_counts, class_weights):
    print(f"{cls:6s} count={count:5d}, weight={weight.item():.4f}")

akiec  count=  294, weight=4.3795
bcc    count=  463, weight=2.7809
bkl    count=  989, weight=1.3019
df     count=  103, weight=12.5007
mel    count= 1002, weight=1.2850
nv     count= 6034, weight=0.2134
vasc   count=  128, weight=10.0592


In [12]:
image_size = 299

train_tfms = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(30),
    transforms.ColorJitter(
        brightness=0.15,
        contrast=0.15,
        saturation=0.10,
        hue=0.03
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_tfms = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


class HAMMetadataDataset(Dataset):
    def __init__(self, dataframe, metadata_array, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.metadata_array = metadata_array
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image = Image.open(row["image_path"]).convert("RGB")
        if self.transform:
            image = self.transform(image)

        meta = torch.tensor(self.metadata_array[idx], dtype=torch.float32)
        label = torch.tensor(row["label"], dtype=torch.long)

        image_id = row["image_id"]
        true_label_text = row["dx"]

        return image, meta, label, image_id, true_label_text

In [13]:
batch_size = 16

train_dataset = HAMMetadataDataset(train_df, train_meta, transform=train_tfms)
val_dataset = HAMMetadataDataset(val_df, val_meta, transform=val_tfms)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

In [14]:
batch = next(iter(train_loader))
images, metas, labels, image_ids, true_labels_text = batch

print(images.shape)
print(metas.shape)
print(labels.shape)
print(image_ids[:3])
print(true_labels_text[:3])

torch.Size([16, 3, 299, 299])
torch.Size([16, 19])
torch.Size([16])
['ISIC_0028345', 'ISIC_0031165', 'ISIC_0028873']
['nv', 'nv', 'nv']


In [15]:
class SoftAttention(nn.Module):
    def __init__(self, in_channels, num_heads=16):
        super().__init__()

        self.num_heads = num_heads

        self.attention_conv = nn.Conv2d(
            in_channels=in_channels,
            out_channels=num_heads,
            kernel_size=1,
            bias=True
        )

        self.gamma = nn.Parameter(torch.zeros(1))

    def forward(self, x):
        # x: [B, C, H, W]
        B, C, H, W = x.shape

        attn = self.attention_conv(x)          # [B, K, H, W]
        attn = attn.view(B, self.num_heads, -1)
        attn = torch.softmax(attn, dim=-1)
        attn = attn.view(B, self.num_heads, H, W)

        attn_map = attn.mean(dim=1, keepdim=True)  # [B, 1, H, W]

        attended = x * attn_map
        attended = self.gamma * attended

        out = torch.cat([x, attended], dim=1)       # [B, 2C, H, W]

        return out, attn_map


class InceptionResNetV2SoftAttentionMetadata(nn.Module):
    def __init__(self, metadata_dim, num_classes=7, pretrained=True):
        super().__init__()

        self.backbone = timm.create_model(
            "inception_resnet_v2",
            pretrained=pretrained,
            features_only=True,
            out_indices=(-1,)
        )

        feature_channels = self.backbone.feature_info.channels()[-1]

        self.soft_attention = SoftAttention(
            in_channels=feature_channels,
            num_heads=16
        )

        self.image_pool = nn.AdaptiveAvgPool2d(1)

        self.metadata_mlp = nn.Sequential(
            nn.Linear(metadata_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),

            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3)
        )

        self.classifier = nn.Sequential(
            nn.Linear(feature_channels * 2 + 64, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )

    def forward(self, image, metadata, return_attention=False):
        feature_map = self.backbone(image)[-1]      # [B, C, H, W]

        feature_map, attn_map = self.soft_attention(feature_map)

        image_feat = self.image_pool(feature_map)
        image_feat = image_feat.flatten(1)

        meta_feat = self.metadata_mlp(metadata)

        feat = torch.cat([image_feat, meta_feat], dim=1)
        logits = self.classifier(feat)

        if return_attention:
            return logits, attn_map

        return logits

In [16]:
model = InceptionResNetV2SoftAttentionMetadata(
    metadata_dim=metadata_dim,
    num_classes=num_classes,
    pretrained=True
).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=3
)

In [17]:
batch = next(iter(train_loader))
images, metas, labels, image_ids, true_labels_text = batch

model.eval()

with torch.no_grad():
    images = images.to(device)
    metas = metas.to(device)
    labels = labels.to(device)

    logits, attn_map = model(images, metas, return_attention=True)
    loss = criterion(logits, labels)

print("logits shape:", logits.shape)
print("attention map shape:", attn_map.shape)
print("loss:", loss.item())

logits shape: torch.Size([16, 7])
attention map shape: torch.Size([16, 1, 8, 8])
loss: 1.86907160282135


In [18]:
def compute_metrics(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "precision_macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
    }

In [19]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()

    total_loss = 0.0
    all_preds = []
    all_labels = []

    for images, metas, labels, _, _ in tqdm(loader, desc="Training"):
        images = images.to(device)
        metas = metas.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        logits = model(images, metas)
        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)

        preds = torch.argmax(logits, dim=1)
        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(labels.detach().cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)

    metrics = compute_metrics(all_labels, all_preds)
    metrics["loss"] = avg_loss

    return metrics


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()

    total_loss = 0.0
    all_preds = []
    all_labels = []
    all_probs = []
    all_image_ids = []

    for images, metas, labels, image_ids, _ in tqdm(loader, desc="Validation"):
        images = images.to(device)
        metas = metas.to(device)
        labels = labels.to(device)

        logits = model(images, metas)
        loss = criterion(logits, labels)

        probs = torch.softmax(logits, dim=1)
        preds = torch.argmax(probs, dim=1)

        total_loss += loss.item() * images.size(0)

        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(labels.detach().cpu().numpy())
        all_probs.extend(probs.detach().cpu().numpy())
        all_image_ids.extend(image_ids)

    avg_loss = total_loss / len(loader.dataset)

    metrics = compute_metrics(all_labels, all_preds)
    metrics["loss"] = avg_loss

    return metrics, np.array(all_labels), np.array(all_preds), np.array(all_probs), all_image_ids

In [20]:
train_metrics = train_one_epoch(model, train_loader, optimizer, criterion, device)
val_metrics, y_true, y_pred, y_probs, image_ids = evaluate(model, val_loader, criterion, device)

print("train:", train_metrics)
print("val:", val_metrics)

Validation: 100%|██████████| 63/63 [00:22<00:00,  2.74it/s]

train: {'accuracy': 0.581493398424498, 'balanced_accuracy': np.float64(0.5609620359306898), 'precision_macro': 0.3647632281169703, 'recall_macro': 0.5609620359306898, 'f1_macro': 0.40271991402387275, 'loss': 1.2235220816851482}
val: {'accuracy': 0.6407185628742516, 'balanced_accuracy': np.float64(0.7179451858884287), 'precision_macro': 0.43662870307521306, 'recall_macro': 0.7179451858884287, 'f1_macro': 0.498984084900269, 'loss': 0.8548146800366704}
